# Global scalar mixed model — specification grid

The whole-brain scalar gate: one value per mouse, `~ genotype * sex * light`, and
the three-way interaction as the primary confirmatory statistic. No multiplicity
problem, no per-region machinery.

## What changed in this version

**1. Sign convention is now CONTROL-POSITIVE, matching the per-region and PLSC
notebooks.** Every contrast is built from the same ±1 `code_map`
(`cre/+ = +1`, `M = +1`, `L = +1`) that the per-region pipeline uses, rather than
being read off treatment-coded dummies. Section 4 asserts the two weight sets are
identical element-by-element, so the three notebooks cannot drift apart again.

> **This flips the sign of every genotype-bearing contrast** relative to earlier
> runs. The three-way goes from **β = −2.62 to β = +2.62** — same estimate, same
> SE, same p, opposite sign. `sex:light`, `light`, and `sex` are unaffected. The
> relation to the old coding is `new = −old` for genotype terms, asserted in
> Section 4 so the change is auditable rather than something you have to trust.

**2. The contrast set is the same as the per-region notebook's** -- 25 built,
the 18 the manuscript reports kept (see `PAPER_CONTRASTS`) -- with identical
names, so the global and per-region tables join directly on `contrast`.

**3. A specification grid replaces the single primary fit.** Four global scalars
crossed with two batch treatments = 8 specifications, each carrying the full
contrast table, an optimizer/start-value check, leave-one-mouse-out over all
mice, and leave-one-batch-out for both batch variables.

| scalar | definition | reads as |
|---|---|---|
| `raw` | `mean_j log(density_j)` | log geometric mean over **regions**; a small nucleus counts as much as VISp |
| `volw` | `sum_j w_j log(density_j)`, `w` = voxel share | log geometric mean over **voxels** |
| `root` | `log(density_root)` | total counts / total volume, straight from the pipeline — an *arithmetic* volume-weighted mean, so it is a genuinely different estimand, not a reweighting of the other two |
| `resid` | `mean_j log(density_j)` after per-region batch BLUPs | the legacy doubly-corrected version, kept for comparison only |

| batch | model |
|---|---|
| `batchRE` | `mixedlm(..., groups=batch)` — batch as a random intercept |
| `nobatch` | `ols(...)` — no batch term at all, i.e. **no IHC-batch removal in any form** |

`raw x batchRE` is the primary. `resid x batchRE` is the double correction this
notebook was originally written to diagnose; it is included so the specification
curve shows it rather than the notebook merely arguing against it.

**4. `norm.sf` replaces `2 * (1 - norm.cdf(...))`** in the contrast table — the
same underflow bug that was fixed elsewhere in the project but survived here.

## What to read

Section 9's specification curve. If the three-way holds sign, rough magnitude,
and significance across all eight specifications and every drop, that is one
sentence in the response letter. If it moves, the grid tells you *which* choice
moves it, which is the useful information.

## 1. Config

Paths point at the same two files the PLSC notebook uses, so the analyses sit on
an identical region set and identical metadata.

In [ ]:
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import norm, t as tdist

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

# ---------------------------------------------------------------------------
# Paths -- repo-relative, so this notebook runs wherever the repository is
# checked out. PROJECT_ROOT is found by walking up from the working directory
# until a folder containing `data/` appears. If you keep the data outside the
# repository, set PROJECT_ROOT by hand and everything below follows.
# ---------------------------------------------------------------------------
def find_project_root(markers=("data", "notebooks")):
    """Nearest ancestor directory containing all of `markers`. Requiring both
    `data/` and `notebooks/` means a stray `data/` folder somewhere on the path
    cannot be mistaken for the repository root."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if all((candidate / m).is_dir() for m in markers):
            return candidate
    raise RuntimeError(
        f"No ancestor of {here} contains {list(markers)}. Set PROJECT_ROOT by hand."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR     = PROJECT_ROOT / "data"
RESULTS_DIR  = PROJECT_ROOT / "results"

COMPILED_DIR = DATA_DIR / "compiled" / "cleaned_output"
MATRIX_CSV   = COMPILED_DIR / "density_by_region_leaves_wide.csv"
INFO_CSV     = DATA_DIR / "mouse_info_template.csv"

# ### CHOICE ### Root density source, for the `root` scalar.
# The leaves matrix does not contain `root` by construction. Point this at
# whatever the density pipeline writes for the whole-brain annotation. If the
# file is absent the `root` specifications are SKIPPED with a message -- they are
# never silently replaced by a proxy, because root density is an arithmetic
# volume-weighted mean and nothing else in this notebook reproduces it.
ROOT_DENSITY_CSV = COMPILED_DIR / "density_root.csv"
ROOT_ACRONYM     = "Root"

OUTPUT_DIR = RESULTS_DIR / "global_model"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REGION_ACRONYM_COL   = "Acronym"
MEASUREMENT_COL_FLAG = "annotation_measurements"

GENOS  = ["cre/+", "cre/cre"]
SEXES  = ["F", "M"]
LIGHTS = ["D", "L"]

# ---- Batch variables ----
IMMUNO_BATCH_COL    = "batch"              # staining/IHC batch; the random intercept
PERFUSION_BATCH_COL = "perfusion_batch"    # sex-nested; used for leave-one-batch-out only

# ---- Specification grid ----
# ### CHOICE ###
# Every scalar is crossed with every batch treatment. `resid` is expensive (one
# LMM per region) -- set RUN_RESIDUALIZED=False to drop those two specs while
# iterating.
RUN_RESIDUALIZED    = True
TRY_VOLUME_WEIGHTED = True
ATLAS_NAME          = "allen_mouse_25um"

PRIMARY_SPEC = "raw_batchRE"

# ---- Sensitivity ----
N_RANDOM_STARTS = 3
OPTIMIZERS      = ["lbfgs", "cg", "powell", "bfgs"]
LOO_ALL_MICE    = True     # leave-one-out over EVERY mouse, not just the thin cell

# ### CHOICE ###
# Reference distribution for contrast p-values. Both are reported. df = n - p_fixed
# for the t version. statsmodels MixedLM has no Satterthwaite/Kenward-Roger, so
# the t column is a CONSERVATIVE BOUND, not a correction -- the exact denominator
# df is not identified here. If a reviewer presses, report both and say so.

FORMULA = "{dv} ~ genotype * sex * light"

## 2. Load

Region selection is **listwise complete** across mice, matching the PLSC
notebook's `CORE` set. The global scalar has to be computed over identical
anatomy for every mouse or "whole-brain level" is confounded with coverage, so
unlike the PLSC and per-region notebooks this one does **not** relax the
inclusion rule. That is deliberate, not an oversight.

In [ ]:
raw = pd.read_csv(MATRIX_CSV, index_col=REGION_ACRONYM_COL)
meas_cols = [c for c in raw.columns if MEASUREMENT_COL_FLAG in c]
assert meas_cols, f"No columns containing '{MEASUREMENT_COL_FLAG}' in {MATRIX_CSV}"

X_raw = raw[meas_cols].copy()
X_raw.columns = [c.replace(f"_{MEASUREMENT_COL_FLAG}", "") for c in X_raw.columns]

info = pd.read_csv(INFO_CSV)
info["id"] = info["id"].astype(str)
info = info.set_index("id")

missing_ids = set(X_raw.columns) - set(info.index)
assert not missing_ids, f"{len(missing_ids)} mice in matrix but not in info: {sorted(missing_ids)[:5]}"
info = info.loc[X_raw.columns].copy()          # align to matrix column order

complete = X_raw.notna().all(axis=1)
print(f"{complete.sum()} of {len(X_raw)} regions complete across {X_raw.shape[1]} mice")
if (~complete).any():
    print(f"  dropped {int((~complete).sum())} incomplete regions "
          f"(first 20): {X_raw.index[~complete].tolist()[:20]}")

X       = X_raw[complete]
PSEUDO  = X.values[X.values > 0].min() / 2
Xl      = np.log(X.values.T + PSEUDO)          # mice x regions
regions = X.index.to_numpy()
print(f"pseudocount = {PSEUDO:.6g}   Xl shape: {Xl.shape} (mice x regions)")

required = {"genotype", "sex", "light", IMMUNO_BATCH_COL}
assert not (required - set(info.columns)), f"info missing: {sorted(required - set(info.columns))}"
assert info[IMMUNO_BATCH_COL].notna().all(), "batch column has missing values -- fill the metadata sheet first"

# Fix category order explicitly so treatment coding -- and therefore the dummy
# NAMES the cell-weight builder relies on -- is known rather than inferred.
info["genotype"] = pd.Categorical(info["genotype"], categories=GENOS)
info["sex"]      = pd.Categorical(info["sex"],      categories=SEXES)
info["light"]    = pd.Categorical(info["light"],    categories=LIGHTS)
info[IMMUNO_BATCH_COL] = info[IMMUNO_BATCH_COL].astype(str)

## 3. Design cells

Printed rather than assumed: the three-way is estimated from all eight cells and
is therefore the contrast most exposed to the smallest one.

In [ ]:
cell_counts = info.groupby(["genotype", "sex", "light"], observed=True).size()
print("Design cells (n):")
print(cell_counts.to_string())
print(f"\ntotal n = {cell_counts.sum()}")

if cell_counts.min() < 4:
    print(f"\n  !! smallest cell {cell_counts.idxmin()} has n={cell_counts.min()}")
    print("     the three-way is most exposed to this -- see Section 8")

for col in (PERFUSION_BATCH_COL, IMMUNO_BATCH_COL):
    if col in info.columns:
        print(f"\n{col} x sex:")
        print(pd.crosstab(info[col], info["sex"]).to_string())
        print(f"{col} x genotype:")
        print(pd.crosstab(info[col], info["genotype"]).to_string())

## 4. Contrast machinery — control-positive, shared with the per-region notebook

**Sign convention: `cre/+ = +1`, `M = +1`, `L = +1`.** A positive
`geno:light` estimate means the light effect is more positive in control than in
MKO. A positive three-way means the sex difference in light response is more
positive in control than in MKO.

Every contrast is a length-8 weight vector over the design cells, built from
`code_map` — the *same construction* as `LMM_per_region_permutation_enrichment`,
not a parallel reimplementation. The weight vector is then projected through this
model's own treatment-coded cell design matrix, so the reference level is
irrelevant to any estimate or SE.

The cell below asserts three things rather than asserting them in prose:

1. the weight vectors reproduce the per-region definitions exactly;
2. split two-ways average to the pooled term and differ by the three-way;
3. every genotype-bearing contrast is the **exact negation** of the old
   `cre/cre − cre/+` coding, which is the audit trail for the sign flip.

In [ ]:
CELLS   = list(product(GENOS, SEXES, LIGHTS))
FACTORS = ("genotype", "sex", "light")

# ### CHOICE ### CONTROL-POSITIVE. Identical to the per-region notebook's code_map.
code_map = {"cre/+": 1, "cre/cre": -1, "F": -1, "M": 1, "D": -1, "L": 1}
PRETTY   = {"cre/+": "control", "cre/cre": "MKO", "L": "light", "D": "dark", "M": "M", "F": "F"}

TERM_FACTORS = {"geno:sex":       ("genotype", "sex"),
                "geno:light":     ("genotype", "light"),
                "sex:light":      ("sex", "light"),
                "geno:sex:light": ("genotype", "sex", "light")}

cell_dict = lambda cell: dict(zip(FACTORS, cell))


def simple_effect_weight(fix, vary_col, ref_val, target_val):
    """+1 at the target cell, -1 at the reference cell, 0 elsewhere."""
    w = []
    for cell in CELLS:
        c = cell_dict(cell)
        if not all(c[k] == v for k, v in fix.items()):
            w.append(0.0)
        elif c[vary_col] == target_val:
            w.append(1.0)
        elif c[vary_col] == ref_val:
            w.append(-1.0)
        else:
            w.append(0.0)
    return np.array(w)


def interaction_weight(term, split=None):
    """Pooled terms divide by 2^(3-order) to average over the omitted factors;
    split terms are saturated within their stratum and are NOT divided."""
    facs = TERM_FACTORS[term]
    if split is not None and set(split) & set(facs):
        raise ValueError(f"cannot split '{term}' by {list(split)} -- already an interacting factor")
    w = []
    for cell in CELLS:
        c = cell_dict(cell)
        if split is not None and any(c[k] != v for k, v in split.items()):
            w.append(0.0)
            continue
        wi = float(np.prod([code_map[c[f]] for f in facs]))
        if split is None:
            wi /= 2 ** (3 - len(facs))
        w.append(wi)
    return np.array(w)


def main_effect_weight(factor):
    """Type-III / emmeans marginal: equal cell weighting despite unequal n."""
    return np.array([code_map[cell_dict(c)[factor]] / 4 for c in CELLS])


# ---- 12 simple effects ----
# ### CHOICE ### `control_vs_MKO`, not `MKO_vs_control`. Under the control-positive
# convention the genotype simple effect is cre/+ minus cre/cre, and the NAME has to
# say so -- the old label paired MKO-minus-control arithmetic with a
# control-positive interaction set, so one output table carried both directions.
SIMPLE_SPECS = [
    ("light",    "D",       "L",     "light_vs_dark",  ("genotype", "sex")),
    ("genotype", "cre/cre", "cre/+", "control_vs_MKO", ("light", "sex")),
    ("sex",      "F",       "M",     "M_vs_F",         ("genotype", "light")),
]
LEVELS = {"genotype": GENOS, "sex": SEXES, "light": LIGHTS}

SIMPLE_CONTRASTS = {}
for vary, ref, tgt, stem, show in SIMPLE_SPECS:
    for a in LEVELS[show[0]]:
        for b in LEVELS[show[1]]:
            SIMPLE_CONTRASTS[f"{stem} | {PRETTY[a]} | {PRETTY[b]}"] = \
                simple_effect_weight({show[0]: a, show[1]: b}, vary, ref, tgt)

# ---- 3 pooled two-ways + the three-way, then 6 split two-ways ----
INTERACTION_CONTRASTS = {k: interaction_weight(k) for k in
                         ["geno:sex", "geno:light", "sex:light", "geno:sex:light"]}
SPLIT_SPECS = [("geno:light", "sex", ["M", "F"]),
               ("geno:sex", "light", ["L", "D"]),
               ("sex:light", "genotype", ["cre/+", "cre/cre"])]
for term, by, lvs in SPLIT_SPECS:
    for lv in lvs:
        INTERACTION_CONTRASTS[f"{term} | {PRETTY[lv]}"] = interaction_weight(term, split={by: lv})

MAIN_EFFECT_CONTRASTS = {f"{f} (main effect)": main_effect_weight(f)
                         for f in ["light", "genotype", "sex"]}

ALL_CONTRASTS = {**{k: ("simple", v)      for k, v in SIMPLE_CONTRASTS.items()},
                 **{k: ("interaction", v) for k, v in INTERACTION_CONTRASTS.items()},
                 **{k: ("main_effect", v) for k, v in MAIN_EFFECT_CONTRASTS.items()}}
assert len(ALL_CONTRASTS) == 25, f"expected 25 contrasts, got {len(ALL_CONTRASTS)}"

# ---- Structural assertions --------------------------------------------------
three_way = INTERACTION_CONTRASTS["geno:sex:light"]
for term, by, lvs in SPLIT_SPECS:
    a = INTERACTION_CONTRASTS[f"{term} | {PRETTY[lvs[0]]}"]
    b = INTERACTION_CONTRASTS[f"{term} | {PRETTY[lvs[1]]}"]
    assert np.allclose((a + b) / 2, INTERACTION_CONTRASTS[term]), \
        f"{term}: split levels do not average to the pooled term"
    assert np.allclose(a - b, three_way) or np.allclose(b - a, three_way), \
        f"{term}: difference of split levels is not the three-way"

# ---- The sign-flip audit ----------------------------------------------------
# Rebuild the OLD (cre/cre - cre/+) three-way and confirm exact negation.
old_three_way = np.array([
    (1 if g == "cre/cre" else -1) * (1 if s == "M" else -1) * (1 if l == "L" else -1)
    for g, s, l in CELLS], float)
assert np.allclose(three_way, -old_three_way), "sign-flip relation is not exact negation"

# Contrasts that carry NO genotype must be untouched by the flip.
for nm in ["sex:light | control", "sex:light | MKO", "light (main effect)", "sex (main effect)"]:
    w = ALL_CONTRASTS[nm][1]
    assert np.isfinite(w).all()

# ---- Restrict to the contrasts the manuscript reports ----------------------
# ### CHOICE ###
# Identical list to the per-region notebook's PAPER_CONTRASTS, so Section 4a's
# cross-check still compares like with like. All 25 are constructed above because
# the structural and sign-flip assertions need the pooled two-ways and both split
# levels; only the reported ones are carried forward.
#
# Each contrast is an independent weighted sum over the eight fitted cell means,
# so dropping one cannot move any other estimate, SE, or p-value.
PAPER_CONTRASTS = [
    "light (main effect)", "genotype (main effect)", "sex (main effect)",
    "geno:sex:light",
    "geno:light | M", "geno:light | F",
    "sex:light | control", "sex:light | MKO",
    "geno:sex | light", "geno:sex | dark",
    "light_vs_dark | control | M", "light_vs_dark | control | F",
    "light_vs_dark | MKO | M",     "light_vs_dark | MKO | F",
    "control_vs_MKO | dark | M",  "control_vs_MKO | dark | F",
    "control_vs_MKO | light | M", "control_vs_MKO | light | F",
]
_missing = [c for c in PAPER_CONTRASTS if c not in ALL_CONTRASTS]
assert not _missing, f"PAPER_CONTRASTS names were not built above: {_missing}"
_n_built = len(ALL_CONTRASTS)
ALL_CONTRASTS = {k: ALL_CONTRASTS[k] for k in PAPER_CONTRASTS}

print(f"{len(ALL_CONTRASTS)} of {_n_built} contrasts reported, CONTROL-POSITIVE "
      f"(cre/+ = +1, M = +1, L = +1)")
print("  three-way weight at (cre/+, M, L) = "
      f"{three_way[CELLS.index(('cre/+','M','L'))]:+.0f}  "
      "-> exact negation of the previous cre/cre-positive coding")
print("  genotype-bearing contrasts FLIP sign vs. earlier runs; "
      "sex:light / light / sex are unchanged.\n")
for group in ("main_effect", "interaction", "simple"):
    print(f"{group}:")
    for k, (ct, _) in ALL_CONTRASTS.items():
        if ct == group:
            print(f"  {k}")

### 4a. Cross-check against the per-region notebook

Optional but worth running once per session. Point `PER_REGION_CONTRAST_CSV` at
the per-region pipeline's `OUT_CSV` and this confirms the two notebooks agree on
the **name** of every contrast. Weight equality is guaranteed by construction
(same `code_map`, same builders); name equality is what actually breaks when one
notebook is edited and the other is not.

In [ ]:
PER_REGION_CONTRAST_CSV = RESULTS_DIR / "per_region" / "regional_contrasts_whole_brain.csv"

if PER_REGION_CONTRAST_CSV.exists():
    _pr = pd.read_csv(PER_REGION_CONTRAST_CSV, usecols=["contrast"])
    pr_names, gl_names = set(_pr["contrast"].unique()), set(ALL_CONTRASTS)
    if pr_names == gl_names:
        print(f"OK  contrast names identical across notebooks ({len(gl_names)} contrasts)")
    else:
        print("!! contrast name mismatch -- the two notebooks have drifted:")
        print(f"   per-region only: {sorted(pr_names - gl_names)}")
        print(f"   global only    : {sorted(gl_names - pr_names)}")
        print("   Most likely cause: one notebook still uses 'MKO_vs_control' "
              "(the pre-flip label).")
else:
    print(f"(per-region table not found at {PER_REGION_CONTRAST_CSV} -- skipping name check)")

## 5. The four global scalars

All four are built here so the specification grid reads from one place.

### CHOICE — these are not interchangeable

- `raw` is an **unweighted** mean over log densities: the log geometric mean of
  regional densities, in which a small hypothalamic nucleus counts exactly as
  much as VISp. A defensible estimand ("the average region"), but most readers
  hear volume weighting when they read "whole-brain cFos".
- `volw` weights each region's log density by its voxel share: the log geometric
  mean over voxels.
- `root` is `log` of the pipeline's root-annotation density, i.e. total detected
  cells over total annotated volume. That is an **arithmetic** volume-weighted
  mean of densities, so it is dominated by high-density regions in a way neither
  geometric version is. It is the number a reader would compute from the raw
  counts, which is exactly why it belongs in the grid.
- `resid` reproduces the doubly batch-corrected scalar the PLSC notebook used to
  build, kept only so the specification curve can show what that choice does.

Whichever is adopted as primary, Methods must state which — "whole-brain cFos"
is ambiguous across all four.

In [ ]:
SCALARS = {}          # name -> column in `info`

# ---- raw ----
info["global_cfos_raw"] = Xl.mean(axis=1)
SCALARS["raw"] = "global_cfos_raw"

# ---- volume-weighted ----
def volume_weights(region_list, atlas_name=ATLAS_NAME):
    """Voxel-count weights per region, normalised to sum to 1. None if unavailable.

    One pass over the annotation with np.unique, then descendant ids are summed
    from the structure table -- the old per-region `np.isin(ann, ids)` loop was
    O(n_regions) full-volume scans and took minutes.
    """
    try:
        from brainglobe_atlasapi import BrainGlobeAtlas
    except ImportError:
        print("  brainglobe_atlasapi not available -- skipping volume weighting")
        return None

    atlas = BrainGlobeAtlas(atlas_name)
    labels, counts = np.unique(atlas.annotation, return_counts=True)
    count_by_id = dict(zip(labels.tolist(), counts.tolist()))

    vox = {}
    for r in region_list:
        try:
            sid = atlas.structures[r]["id"]
        except KeyError:
            print(f"  '{r}' not in atlas -- skipping volume weighting entirely")
            return None
        ids = [v["id"] for v in atlas.structures.values() if sid in v["structure_id_path"]]
        vox[r] = sum(count_by_id.get(i, 0) for i in ids)

    w = np.array([vox[r] for r in region_list], float)
    if w.sum() == 0:
        return None
    if (w == 0).any():
        zero = [r for r, v in zip(region_list, w) if v == 0]
        print(f"  !! {len(zero)} region(s) have zero annotated voxels and get zero "
              f"weight: {zero[:10]}")
    return w / w.sum()


VOL_W = volume_weights(regions) if TRY_VOLUME_WEIGHTED else None
if VOL_W is not None:
    info["global_cfos_volw"] = Xl @ VOL_W
    SCALARS["volw"] = "global_cfos_volw"
    print(f"volume weights: range {VOL_W.min():.2e} to {VOL_W.max():.2e} "
          f"({VOL_W.max()/VOL_W.min():.0f}x), sum = {VOL_W.sum():.6f}")

# ---- root density ----
def load_root_density(path, acronym=ROOT_ACRONYM):
    """log(root density + PSEUDO) per mouse, aligned to `info.index`. None if absent."""
    if not Path(path).exists():
        print(f"  root density file not found at {path} -- root specs SKIPPED")
        return None
    rd = pd.read_csv(path, index_col=REGION_ACRONYM_COL)
    cols = [c for c in rd.columns if MEASUREMENT_COL_FLAG in c]
    if cols:
        rd = rd[cols]
        rd.columns = [c.replace(f"_{MEASUREMENT_COL_FLAG}", "") for c in rd.columns]
    hit = [a for a in rd.index if str(a).strip().lower() == acronym.lower()]
    if not hit:
        print(f"  '{acronym}' not present in {Path(path).name} -- root specs SKIPPED")
        return None
    vals = rd.loc[hit[0]]
    missing = set(info.index) - set(vals.index)
    if missing:
        print(f"  root density missing for {len(missing)} mice -- root specs SKIPPED")
        return None
    v = pd.to_numeric(vals.reindex(info.index), errors="coerce")
    if v.isna().any():
        print(f"  root density non-numeric/NaN for {int(v.isna().sum())} mice -- root specs SKIPPED")
        return None
    return np.log(v.values + PSEUDO)


root_vals = load_root_density(ROOT_DENSITY_CSV)
if root_vals is not None:
    info["global_cfos_root"] = root_vals
    SCALARS["root"] = "global_cfos_root"
    print(f"root density loaded: log range {root_vals.min():.2f} to {root_vals.max():.2f}")

# ---- batch-residualized (legacy comparison) ----
def residualize_batch(Xl_, info_, region_list):
    """Per-region LMM with immuno batch as a random intercept; subtract only the
    batch BLUP. Reproduced ONLY to regenerate the legacy doubly-corrected scalar."""
    Xr, failed = Xl_.copy(), []
    for j, r in enumerate(region_list):
        dd = info_.copy()
        dd["y"] = Xl_[:, j]
        try:
            m = smf.mixedlm("y ~ genotype * sex * light", dd,
                            groups=dd[IMMUNO_BATCH_COL]).fit()
            blup = dd[IMMUNO_BATCH_COL].map(
                {k: float(np.asarray(v).ravel()[0]) for k, v in m.random_effects.items()})
            Xr[:, j] = Xl_[:, j] - blup.values
        except Exception:
            failed.append(r)
    if failed:
        print(f"  {len(failed)} regions failed to converge; left un-residualized")
    return Xr


if RUN_RESIDUALIZED:
    Xa = residualize_batch(Xl, info, regions)
    info["global_cfos_resid"] = Xa.mean(axis=1)
    SCALARS["resid"] = "global_cfos_resid"

print(f"\nscalars available: {list(SCALARS)}")
print()
print(info.groupby(["genotype", "sex", "light"], observed=True)[list(SCALARS.values())]
        .agg(["count", "mean"]).round(3).to_string())

print("\nCorrelations between scalars (they are NOT the same quantity):")
print(info[list(SCALARS.values())].corr().round(4).to_string())

## 6. Fitting layer

One function for both batch treatments so the grid, the optimizer check, and the
sensitivity loops all go through identical code. Contrasts are reconstructed from
the eight fitted cell means, so the treatment coding of the fit never touches any
estimate or SE.

In [ ]:
def cell_weight_treatment(fe_index, geno, sex, light):
    """Linear combination of treatment-coded fixed effects reconstructing one
    design cell's fitted mean. Names are asserted rather than assumed."""
    g = 1 if geno  == "cre/cre" else 0
    s = 1 if sex   == "M"       else 0
    l = 1 if light == "L"       else 0
    terms = {"Intercept": 1.0,
             "genotype[T.cre/cre]": g,
             "sex[T.M]": s,
             "light[T.L]": l,
             "genotype[T.cre/cre]:sex[T.M]": g * s,
             "genotype[T.cre/cre]:light[T.L]": g * l,
             "sex[T.M]:light[T.L]": s * l,
             "genotype[T.cre/cre]:sex[T.M]:light[T.L]": g * s * l}
    missing = set(terms) - set(fe_index)
    if missing:
        raise KeyError(f"fitted model is missing expected terms {sorted(missing)}; "
                       f"got {list(fe_index)}")
    w = pd.Series(0.0, index=fe_index)
    for k, v in terms.items():
        w[k] = float(v)
    return w


class Fit:
    """Uniform wrapper over mixedlm / ols.

    `raw` keeps the underlying statsmodels result. It is needed for
    `start_params`: MixedLM expects the PACKED parameter vector (fixed effects
    followed by the covariance parameters), so perturbing `fe_params` alone
    produces a vector of the wrong length.
    """
    __slots__ = ("fe", "cov", "converged", "llf", "group_var", "n", "summary",
                 "batch", "raw")

    def __init__(self, fe, cov, converged, llf, group_var, n, summary, batch, raw):
        self.fe, self.cov, self.converged = fe, cov, converged
        self.llf, self.group_var, self.n = llf, group_var, n
        self.summary, self.batch, self.raw = summary, batch, raw


def fit_model(dv, data, batch="batchRE", method="lbfgs", start_params=None, reml=True):
    """batch='batchRE' -> mixedlm with immuno batch as random intercept.
       batch='nobatch'  -> plain OLS, no batch term in any form."""
    formula = FORMULA.format(dv=dv)
    if batch == "batchRE":
        model = smf.mixedlm(formula, data, groups=data[IMMUNO_BATCH_COL])
        kw = {"reml": reml, "method": method}
        if start_params is not None:
            kw["start_params"] = start_params
        f = model.fit(**kw)
        fe = f.fe_params
        cov = f.cov_params().loc[fe.index, fe.index]
        gv = float(np.asarray(f.cov_re)[0, 0])
        conv = bool(f.converged)
    elif batch == "nobatch":
        f = smf.ols(formula, data).fit()
        fe = f.params
        cov = f.cov_params()
        gv = np.nan
        conv = True          # OLS is closed-form; no optimizer to fail
    else:
        raise ValueError(batch)
    return Fit(fe, cov, conv, float(f.llf), gv, len(data), str(f.summary()), batch, f)


def cell_design(fe_index):
    """8 x k matrix; row i reconstructs CELLS[i] in this fit's coefficient space."""
    return np.array([cell_weight_treatment(fe_index, *c).values for c in CELLS])


def contrast_table(fit, label, spec=None):
    """Wald tests for every reported contrast, against BOTH a normal and a t reference."""
    fe, cov = fit.fe, fit.cov
    M  = cell_design(fe.index)
    df = max(fit.n - len(fe), 1)

    rows = []
    for name, (ctype, w) in ALL_CONTRASTS.items():
        cvec = w @ M
        est  = float(cvec @ fe.values)
        var  = float(cvec @ cov.values @ cvec)
        se   = np.sqrt(var) if var > 0 else np.nan
        stat = est / se if (se and np.isfinite(se)) else np.nan
        ok   = np.isfinite(stat)
        rows.append({
            "spec": spec or label, "model": label, "contrast": name,
            "contrast_type": ctype, "estimate": est, "se": se, "stat": stat,
            # ### FIX ### norm.sf, not 2*(1 - norm.cdf(|z|)): the latter underflows
            # to EXACTLY 0 once |z| exceeds ~8.3, making -log10(p) infinite.
            "p_normal": float(2 * norm.sf(abs(stat))) if ok else np.nan,
            "p_t":      float(2 * tdist.sf(abs(stat), df)) if ok else np.nan,
            "df_t": df, "converged": fit.converged, "n": fit.n})
    return pd.DataFrame(rows)


# ---- Sanity check: weight vectors reduce to the expected forms ----
_probe = fit_model(SCALARS["raw"], info, batch="batchRE")
_M = cell_design(_probe.fe.index)
print("Weight vectors in treatment-coded coefficient space:\n")
for _k in ["geno:sex:light", "light_vs_dark | control | F",
           "light (main effect)", "genotype (main effect)"]:
    _cv = pd.Series(ALL_CONTRASTS[_k][1] @ _M, index=_probe.fe.index)
    print(f"  {_k:34s} {dict((a, round(b, 3)) for a, b in _cv[_cv.abs() > 1e-12].items())}")
print("\n  ^ the three-way should reduce to the single dummy with coefficient -1,")
print("    which IS the control-positive flip made explicit.")

## 7. Specification grid

Every scalar crossed with every batch treatment. One contrast table per
specification, concatenated into `contrasts`.

In [ ]:
SPEC_GRID = [(f"{s}_{b}", dv, b)
             for s, dv in SCALARS.items()
             for b in ("batchRE", "nobatch")]

fits, tabs = {}, []
print(f"Fitting {len(SPEC_GRID)} specifications\n")
for spec, dv, batch in SPEC_GRID:
    try:
        f = fit_model(dv, info, batch=batch)
    except Exception as e:
        print(f"  {spec:26s} FAILED: {type(e).__name__}: {str(e)[:90]}")
        continue
    fits[spec] = f
    tabs.append(contrast_table(f, spec, spec=spec))
    tw = tabs[-1].query("contrast == 'geno:sex:light'").iloc[0]
    gv = "  -" if not np.isfinite(f.group_var) else f"{f.group_var:.6f}"
    print(f"  {spec:26s} three-way = {tw['estimate']:+7.3f}  se {tw['se']:.3f}  "
          f"p {tw['p_normal']:.5f}   GroupVar {gv:>9s}   converged {f.converged}")

contrasts = pd.concat(tabs, ignore_index=True)
print(f"\n{len(contrasts)} rows = {contrasts['spec'].nunique()} specs x "
      f"{len(ALL_CONTRASTS)} contrasts")

print(f"\n\nPRIMARY SPECIFICATION: {PRIMARY_SPEC}")
print("=" * 78)
print(fits[PRIMARY_SPEC].summary)
print(f"\nGroup Var = {fits[PRIMARY_SPEC].group_var:.6f}")
print("  Compare against the ~0.0002 the batch-residualized specs report. A")
print("  substantively non-zero value here confirms that batch variance exists in")
print("  the raw scalar and was simply being removed twice before.")

In [ ]:
# Full contrast table for the primary specification.
_p = contrasts[contrasts.spec == PRIMARY_SPEC]
print(f"ALL REPORTED CONTRASTS -- {PRIMARY_SPEC}   (control-positive: cre/+ = +1, M = +1, L = +1)")
print("=" * 100)
print(_p[["contrast_type", "contrast", "estimate", "se", "stat", "p_normal", "p_t"]]
      .round(4).to_string(index=False))

## 8. Robustness, per specification

Three loops, all targeting the three-way:

- **(i) optimizer and start values** — four optimizers plus random starts.
  Non-converged fits are excluded from the pass/fail range but still reported.
  The earlier `UNSTABLE` verdict on this model was an artifact of pooling
  non-converged `cg`/`bfgs` fits into the log-likelihood range.
- **(ii) leave-one-batch-out** — for both the immuno batch and the perfusion
  batch. Drops that would cost a design cell entirely are recorded as such
  rather than fitted.
- **(iii) leave-one-mouse-out** — over **every** mouse, not just the thin
  `(cre/cre, F, L)` cell. That cell is still flagged separately in the output,
  since those are the drops a reviewer will ask about, but a full LOO is cheap
  at n=40 and tells you whether leverage is concentrated anywhere else.

In [ ]:
def optimizer_check(dv, data, batch, spec, n_random_starts=N_RANDOM_STARTS, seed=0):
    """(i) Optimizer / start-value robustness. OLS specs are closed-form and are
    reported as trivially stable rather than skipped."""
    rows = []
    if batch == "nobatch":
        f = fit_model(dv, data, batch=batch)
        M = cell_design(f.fe.index); w = ALL_CONTRASTS["geno:sex:light"][1] @ M
        rows.append({"spec": spec, "run": "ols_closed_form", "converged": True,
                     "llf": f.llf, "group_var": np.nan,
                     "three_way_coef": float(w @ f.fe.values),
                     "three_way_p": float(2 * norm.sf(abs(
                         (w @ f.fe.values) / np.sqrt(w @ f.cov.values @ w))))})
        return pd.DataFrame(rows)

    rng = np.random.default_rng(seed)
    w8  = ALL_CONTRASTS["geno:sex:light"][1]

    def record(run, f):
        M = cell_design(f.fe.index); w = w8 @ M
        est = float(w @ f.fe.values); se = float(np.sqrt(w @ f.cov.values @ w))
        rows.append({"spec": spec, "run": run, "converged": f.converged, "llf": f.llf,
                     "group_var": f.group_var, "three_way_coef": est,
                     "three_way_p": float(2 * norm.sf(abs(est / se)))})

    for opt in OPTIMIZERS:
        try:
            record(f"default_start_{opt}", fit_model(dv, data, batch=batch, method=opt))
        except Exception as e:
            rows.append({"spec": spec, "run": f"default_start_{opt}", "converged": False,
                         "llf": np.nan, "group_var": np.nan, "three_way_coef": np.nan,
                         "three_way_p": np.nan, "error": str(e)[:100]})

    base = fit_model(dv, data, batch=batch, method="lbfgs")
    base_packed = np.asarray(base.raw.params, dtype=float)   # fe params + cov params
    for i in range(n_random_starts):
        start = base_packed + rng.normal(0, 0.5, size=base_packed.shape)
        try:
            record(f"random_start_{i}", fit_model(dv, data, batch=batch,
                                                  method="lbfgs", start_params=start))
        except Exception as e:
            rows.append({"spec": spec, "run": f"random_start_{i}", "converged": False,
                         "llf": np.nan, "group_var": np.nan, "three_way_coef": np.nan,
                         "three_way_p": np.nan, "error": str(e)[:100]})
    return pd.DataFrame(rows)


opt_rows, opt_verdicts = [], []
for spec, dv, batch in SPEC_GRID:
    if spec not in fits:
        continue
    tab = optimizer_check(dv, info, batch, spec)
    opt_rows.append(tab)
    ok = tab[tab["converged"] & tab["llf"].notna()]
    llf_r = (ok["llf"].max() - ok["llf"].min()) if len(ok) else np.nan
    p_r   = (ok["three_way_p"].max() - ok["three_way_p"].min()) if len(ok) else np.nan
    c_r   = (ok["three_way_coef"].max() - ok["three_way_coef"].min()) if len(ok) else np.nan
    opt_verdicts.append({"spec": spec, "n_converged": len(ok), "n_runs": len(tab),
                         "llf_range": llf_r, "coef_range": c_r, "p_range": p_r,
                         "verdict": "STABLE" if (np.isfinite(llf_r) and llf_r < 0.01
                                                 and p_r < 0.01) else "CHECK"})

optimizer_results = pd.concat(opt_rows, ignore_index=True)
optimizer_summary = pd.DataFrame(opt_verdicts)
print("(i) OPTIMIZER / START-VALUE ROBUSTNESS  (ranges over CONVERGED fits only)")
print(optimizer_summary.round(6).to_string(index=False))
print("\n  Non-converged fits are reported in `optimizer_results` but excluded from")
print("  the ranges -- pooling them is what produced the earlier false UNSTABLE call.")

In [ ]:
def sensitivity(dv, data, batch, spec):
    """(ii) leave-one-batch-out for every batch variable, and (iii) leave-one-mouse-out."""
    rows = []
    w8 = ALL_CONTRASTS["geno:sex:light"][1]

    def record(sub, analysis, dropped, note=""):
        if sub.groupby(["genotype", "sex", "light"], observed=True).ngroups < 8:
            rows.append({"spec": spec, "analysis": analysis, "dropped": str(dropped),
                         "n": len(sub), "converged": False, "three_way_coef": np.nan,
                         "se": np.nan, "three_way_p": np.nan, "note": "lost a design cell"})
            return
        try:
            f = fit_model(dv, sub, batch=batch)
            M = cell_design(f.fe.index); w = w8 @ M
            est = float(w @ f.fe.values); se = float(np.sqrt(w @ f.cov.values @ w))
            rows.append({"spec": spec, "analysis": analysis, "dropped": str(dropped),
                         "n": len(sub), "converged": f.converged, "three_way_coef": est,
                         "se": se, "three_way_p": float(2 * norm.sf(abs(est / se))),
                         "note": note})
        except Exception as e:
            rows.append({"spec": spec, "analysis": analysis, "dropped": str(dropped),
                         "n": len(sub), "converged": False, "three_way_coef": np.nan,
                         "se": np.nan, "three_way_p": np.nan,
                         "note": f"{type(e).__name__}: {str(e)[:70]}"})

    record(data, "full", "-")

    # (iii) leave-one-mouse-out, every mouse
    if LOO_ALL_MICE:
        for mouse in data.index:
            r = data.loc[mouse]
            cell = f"{r['genotype']}/{r['sex']}/{r['light']}"
            record(data.drop(index=mouse), "drop_mouse", mouse, note=f"cell {cell}")

    # (ii) leave-one-batch-out, both batch variables
    for col in (IMMUNO_BATCH_COL, PERFUSION_BATCH_COL):
        if col not in data.columns:
            continue
        for b in pd.Series(data[col]).dropna().unique():
            record(data[data[col] != b], f"drop_{col}", b)
    return pd.DataFrame(rows)


sens = pd.concat([sensitivity(dv, info, batch, spec)
                  for spec, dv, batch in SPEC_GRID if spec in fits],
                 ignore_index=True)

print("(ii)+(iii) SENSITIVITY -- summary of the three-way across drops, per spec")
print("=" * 100)
drops = sens[sens.analysis != "full"]
summ = (drops.groupby(["spec", "analysis"])
        .agg(n_fits=("three_way_coef", "size"),
             n_usable=("three_way_coef", lambda x: int(np.isfinite(x).sum())),
             coef_min=("three_way_coef", "min"), coef_max=("three_way_coef", "max"),
             p_max=("three_way_p", "max"),
             n_p_over_05=("three_way_p", lambda x: int((x >= 0.05).sum())))
        .reset_index())
print(summ.round(4).to_string(index=False))
print("\n  n_p_over_05 is the number of drops under which the three-way loses")
print("  significance. Zero across the board is the single clean sentence for the")
print("  response letter; anything else names the animal or batch to disclose.")

# The drops a reviewer will ask about, spelled out for the primary spec.
lev = sens[(sens.spec == PRIMARY_SPEC) & (sens.analysis == "drop_mouse")
           & sens["note"].astype(str).str.contains("cre/cre/F/L", na=False)]
if len(lev):
    print(f"\nLeave-one-out within the thin (cre/cre, F, L) cell -- {PRIMARY_SPEC}:")
    print(lev[["dropped", "n", "three_way_coef", "se", "three_way_p"]]
          .round(5).to_string(index=False))

worst = drops[(drops.spec == PRIMARY_SPEC) & np.isfinite(drops.three_way_p)] \
        .nlargest(5, "three_way_p")
if len(worst):
    print(f"\nFive most influential drops for {PRIMARY_SPEC} (largest resulting p):")
    print(worst[["analysis", "dropped", "note", "n", "three_way_coef", "three_way_p"]]
          .round(5).to_string(index=False))

## 9. Specification curve

The figure to look at. Each specification's three-way estimate with its 95% CI,
plus the full range across every leave-one-out and leave-one-batch-out fit drawn
behind it. A result that survives here is robust to the scalar definition, to the
batch treatment, and to any single animal or batch.

In [ ]:
key = (contrasts[contrasts.contrast == "geno:sex:light"]
       .set_index("spec")
       .reindex([s for s, _, _ in SPEC_GRID if s in fits]))
key["ci_lo"] = key["estimate"] - 1.96 * key["se"]
key["ci_hi"] = key["estimate"] + 1.96 * key["se"]

rng_by_spec = (drops[np.isfinite(drops.three_way_coef)]
               .groupby("spec")["three_way_coef"].agg(["min", "max"]))

fig, axes = plt.subplots(2, 1, figsize=(9.5, 7), sharex=True,
                         gridspec_kw={"height_ratios": [2.2, 1]})
x = np.arange(len(key))

ax = axes[0]
for i, spec in enumerate(key.index):
    if spec in rng_by_spec.index:
        ax.plot([i, i], [rng_by_spec.loc[spec, "min"], rng_by_spec.loc[spec, "max"]],
                color="#BBBBBB", lw=6, solid_capstyle="butt", zorder=1)
ax.errorbar(x, key["estimate"], yerr=[key["estimate"] - key["ci_lo"],
                                      key["ci_hi"] - key["estimate"]],
            fmt="o", ms=6, capsize=3, lw=1.4, color="black", zorder=3)
ax.scatter(x, key["estimate"], s=70, zorder=4, edgecolor="black", lw=.6,
           c=["#C44E52" if s == PRIMARY_SPEC else "#4C72B0" for s in key.index])
ax.axhline(0, color="gray", lw=.9, ls="--")
ax.set_ylabel("geno:sex:light estimate (log units)")
ax.set_title("Specification curve, three-way interaction\n"
             "black = point estimate +/- 95% CI;  grey band = full range across "
             "leave-one-out and leave-one-batch-out", fontsize=9)

ax = axes[1]
ax.scatter(x, -np.log10(key["p_normal"]), s=55, edgecolor="black", lw=.6,
           c=["#C44E52" if s == PRIMARY_SPEC else "#4C72B0" for s in key.index])
ax.axhline(-np.log10(0.05), color="gray", lw=.9, ls="--")
ax.text(len(x) - .5, -np.log10(0.05), " p = 0.05", fontsize=7, va="bottom",
        ha="right", color="gray")
ax.set_ylabel("$-\\log_{10}$ p")
ax.set_xticks(x)
ax.set_xticklabels(key.index, rotation=40, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "specification_curve_three_way.png", dpi=200)
plt.show()

print("THREE-WAY ACROSS SPECIFICATIONS  (control-positive)")
print(key[["estimate", "se", "ci_lo", "ci_hi", "stat", "p_normal", "p_t",
           "converged", "n"]].round(5).to_string())

signs = np.sign(key["estimate"].dropna())
print(f"\n  sign agreement across specs : "
      f"{'ALL SAME' if signs.nunique() == 1 else 'DISAGREEMENT -- investigate'}")
print(f"  specs with p_normal < 0.05  : {int((key['p_normal'] < 0.05).sum())} / {len(key)}")
print(f"  specs with p_t      < 0.05  : {int((key['p_t'] < 0.05).sum())} / {len(key)}")

In [ ]:
print("Sex-split geno:light -- the crossover that explains the null pooled geno:light:")
print(contrasts[contrasts.contrast.str.startswith("geno:light |")]
      [["spec", "contrast", "estimate", "se", "p_normal", "p_t"]]
      .round(4).to_string(index=False))

print("\n\nsex:light within each genotype -- the crossover stated directly.")
print("These carry NO genotype term and are therefore UNCHANGED by the sign flip:")
print(contrasts[contrasts.contrast.str.startswith("sex:light |")]
      [["spec", "contrast", "estimate", "se", "p_normal", "p_t"]]
      .round(4).to_string(index=False))

## 10. Export

In [ ]:
contrasts.to_csv(OUTPUT_DIR / "global_model_contrasts.csv", index=False)
optimizer_results.to_csv(OUTPUT_DIR / "global_model_optimizer_runs.csv", index=False)
optimizer_summary.to_csv(OUTPUT_DIR / "global_model_optimizer_summary.csv", index=False)
sens.to_csv(OUTPUT_DIR / "global_model_sensitivity.csv", index=False)
key.to_csv(OUTPUT_DIR / "global_model_specification_curve.csv")

(OUTPUT_DIR / "global_model_summaries.txt").write_text(
    "\n\n\n".join(f"{'='*78}\n{spec}\n{'='*78}\n{f.summary}"
                  for spec, f in fits.items()))

keep = [c for c in ["genotype", "sex", "light", IMMUNO_BATCH_COL, PERFUSION_BATCH_COL,
                    "litter"] + list(SCALARS.values()) if c in info.columns]
info[keep].to_csv(OUTPUT_DIR / "global_cfos_values.csv")

print(f"Wrote outputs to {OUTPUT_DIR.resolve()}")
for f_ in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f_.name}")

print("\n" + "=" * 78)
print("REPORTING CHECKLIST")
print("=" * 78)
print(f"  sign convention      CONTROL-POSITIVE (cre/+ = +1, M = +1, L = +1)")
print(f"                       genotype-bearing contrasts are NEGATED relative to")
print(f"                       any run before this one -- check the manuscript text")
print(f"  primary spec         {PRIMARY_SPEC}")
print(f"  scalars in grid      {list(SCALARS)}")
print(f"  specifications       {len(fits)}")
print(f"  mice                 {len(info)}  (smallest cell n = {cell_counts.min()})")
print(f"  regions in scalar    {len(regions)} (listwise complete)")
print(f"  pseudocount          {PSEUDO:.6g}")
print(f"  LOO fits             {int((sens.analysis == 'drop_mouse').sum())}")
print(f"  leave-batch-out fits {int(sens.analysis.str.startswith('drop_').sum() - (sens.analysis == 'drop_mouse').sum())}")
print("\n  State in Methods WHICH scalar is primary: raw/volw/root are three")
print("  different estimands, not three estimates of one thing.")